# Stock Return Predictor — MAE-Optimized Ensemble

5-day and 10-day return prediction with a validation-learned persistence component.


## Setup


In [ ]:
import warnings
from io import StringIO
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import yfinance as yf
import lightgbm as lgb

from scipy.optimize import nnls, minimize
from scipy.stats import spearmanr

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, SGDRegressor
from sklearn.svm import LinearSVR
from sklearn.kernel_approximation import Nystroem
from sklearn.pipeline import make_pipeline
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score,
)

warnings.filterwarnings('ignore')
np.random.seed(42)


## 1. S&P 500 tickers


In [ ]:
url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
headers = {'User-Agent': 'Mozilla/5.0'}

response = requests.get(url, headers=headers, timeout=30)
response.raise_for_status()

sp500 = pd.read_html(StringIO(response.text))[0]

tickers = (
    sp500['Symbol']
    .str.replace('.', '-', regex=False)
    .tolist()
)

sector_lookup = sp500[['Symbol', 'GICS Sector']].copy()
sector_lookup['Symbol'] = sector_lookup['Symbol'].str.replace('.', '-', regex=False)
sector_lookup = sector_lookup.rename(
    columns={'Symbol': 'Ticker', 'GICS Sector': 'Sector'}
)

print('Companies:', len(tickers))


## 2. Historical data and manual forecast refresh

The end date follows the day you run the notebook. The daily cache has a date in its filename, so yesterday's data is not silently reused. Re-running **all cells** trains models again and rewrites the six CSV files. Run after US market close for completed daily bars. The UI's latest-price panel alone does **not** refresh model forecasts.


In [ ]:
start_date = '2015-01-01'
# Yahoo's end date is exclusive. Run after the US market closes for a completed daily bar.
end_date = (pd.Timestamp.now(tz='UTC').normalize() + pd.Timedelta(days=1)).strftime('%Y-%m-%d')
FORCE_DATA_REFRESH = False  # Set True to redownload during the same day.
cache_file = Path(f'sp500_raw_{end_date}.pkl')

if cache_file.exists() and not FORCE_DATA_REFRESH:
    data = pd.read_pickle(cache_file)
    print('Loaded cached data.')
else:
    data = yf.download(
        tickers,
        start=start_date,
        end=end_date,
        auto_adjust=False,
        group_by='column',
        threads=True,
        progress=False,
    )
    data.to_pickle(cache_file)
    print('Downloaded and cached data.')

print('Raw shape:', data.shape)


## 3. Prepare data


In [ ]:
long_data = data.stack(level='Ticker').reset_index()

price_columns = ['Open', 'High', 'Low', 'Close', 'Volume']
long_data = long_data.dropna(subset=price_columns)
long_data = long_data.sort_values(['Ticker', 'Date']).reset_index(drop=True)

print('Rows:', len(long_data))
print('Companies:', long_data['Ticker'].nunique())
print('Dates:', long_data['Date'].min(), 'to', long_data['Date'].max())


## 4. Features


In [ ]:
def create_features(group):
    group = group.sort_values('Date').copy()
    close = group['Close']

    group['Daily_Return'] = close.pct_change(fill_method=None)
    group['Return_5d'] = close.pct_change(5, fill_method=None)
    group['Return_20d'] = close.pct_change(20, fill_method=None)

    sma20 = close.rolling(20).mean()
    sma50 = close.rolling(50).mean()
    group['Price_SMA20_Distance'] = (close - sma20) / sma20
    group['Price_SMA50_Distance'] = (close - sma50) / sma50

    group['Volatility_20'] = group['Daily_Return'].rolling(20).std()
    group['High_Low_Range_Pct'] = (group['High'] - group['Low']) / close
    group['Volume_Ratio_20'] = group['Volume'] / (group['Volume'].rolling(20).mean() + 1e-8)

    delta = close.diff()
    gain = delta.where(delta > 0, 0).rolling(14).mean()
    loss = -delta.where(delta < 0, 0).rolling(14).mean()
    rs = gain / (loss + 1e-8)
    group['RSI_14'] = 100 - (100 / (1 + rs))

    std20 = close.rolling(20).std()
    upper = sma20 + 2 * std20
    lower = sma20 - 2 * std20
    group['BB_Percent_B'] = (close - lower) / (upper - lower + 1e-8)

    group['ROC_10'] = close.pct_change(10, fill_method=None)

    ema12 = close.ewm(span=12, adjust=False).mean()
    ema26 = close.ewm(span=26, adjust=False).mean()
    group['EMA_Spread_Pct'] = (ema12 - ema26) / close

    previous_close = close.shift(1)
    true_range = pd.concat(
        [
            group['High'] - group['Low'],
            (group['High'] - previous_close).abs(),
            (group['Low'] - previous_close).abs(),
        ],
        axis=1,
    ).max(axis=1)
    group['ATR_14_Pct'] = true_range.rolling(14).mean() / close

    high_252 = group['High'].rolling(252).max()
    low_252 = group['Low'].rolling(252).min()
    group['Position_52w'] = (close - low_252) / (high_252 - low_252 + 1e-8)

    return group

feature_frames = [
    create_features(rows)
    for _, rows in long_data.groupby('Ticker')
]
long_data = pd.concat(feature_frames, ignore_index=True)

# Relative to the whole market on the same day
market_stats = (
    long_data.groupby('Date')['Daily_Return']
    .agg(['mean', 'std'])
    .rename(columns={'mean': 'Market_Mean', 'std': 'Market_Std'})
)
long_data = long_data.merge(market_stats, on='Date', how='left')
long_data['Relative_Return_Z'] = (
    (long_data['Daily_Return'] - long_data['Market_Mean']) /
    (long_data['Market_Std'] + 1e-8)
)
long_data = long_data.drop(columns=['Market_Mean', 'Market_Std'])

# Relative to the stock's sector on the same day
long_data = long_data.merge(sector_lookup, on='Ticker', how='left')
sector_stats = (
    long_data.groupby(['Date', 'Sector'])['Daily_Return']
    .agg(['mean', 'std'])
    .rename(columns={'mean': 'Sector_Mean', 'std': 'Sector_Std'})
)
long_data = long_data.merge(sector_stats, on=['Date', 'Sector'], how='left')
long_data['Relative_Return_Sector_Z'] = (
    (long_data['Daily_Return'] - long_data['Sector_Mean']) /
    (long_data['Sector_Std'] + 1e-8)
)
long_data = long_data.drop(columns=['Sector_Mean', 'Sector_Std'])

print('Feature engineering complete.')


## 5. Targets


In [ ]:
HORIZONS = [5, 10]

feature_columns = [
    'Daily_Return',
    'Return_5d',
    'Return_20d',
    'Price_SMA20_Distance',
    'Price_SMA50_Distance',
    'Volatility_20',
    'High_Low_Range_Pct',
    'Volume_Ratio_20',
    'RSI_14',
    'BB_Percent_B',
    'ROC_10',
    'EMA_Spread_Pct',
    'ATR_14_Pct',
    'Position_52w',
    'Relative_Return_Z',
    'Relative_Return_Sector_Z',
]

for horizon in HORIZONS:
    future_col = f'Future_Close_{horizon}d'
    target_col = f'Target_Return_{horizon}d'

    long_data[future_col] = long_data.groupby('Ticker')['Close'].shift(-horizon)
    long_data[target_col] = long_data[future_col] / long_data['Close'] - 1

long_data[feature_columns] = long_data[feature_columns].replace([np.inf, -np.inf], np.nan)

print('Horizons:', HORIZONS)
print('Features:', len(feature_columns))


## 6. Helpers


In [ ]:
TRAIN_END = '2023-01-01'
VALIDATION_END = '2025-01-01'

LIGHTGBM_PRIORITY = 1.15
BLEND_PRIOR_STRENGTH = 0.00005


def evaluate_predictions(name, predicted_return, data, actual_return, future_col, baseline_mae):
    predicted_return = np.asarray(predicted_return)
    actual_return = np.asarray(actual_return)

    current_price = data['Close'].to_numpy()
    actual_price = data[future_col].to_numpy()
    predicted_price = current_price * (1 + predicted_return)

    mae = mean_absolute_error(actual_price, predicted_price)
    pred_std = np.std(predicted_return) * 100

    if pred_std > 0:
        direction = (np.sign(predicted_return) == np.sign(actual_return)).mean() * 100
        pearson_ic = np.corrcoef(predicted_return, actual_return)[0, 1]
        spearman_ic = spearmanr(predicted_return, actual_return).statistic
    else:
        direction = np.nan
        pearson_ic = 0.0
        spearman_ic = 0.0

    return {
        'Model': name,
        'Price MAE ($)': mae,
        'Price RMSE ($)': mean_squared_error(actual_price, predicted_price) ** 0.5,
        'Price MAPE (%)': mean_absolute_percentage_error(actual_price, predicted_price) * 100,
        'Price R²': r2_score(actual_price, predicted_price),
        'Direction Acc (%)': direction,
        'Pearson IC': pearson_ic,
        'Spearman IC': spearman_ic,
        'MAE vs Baseline ($)': baseline_mae - mae,
        'Pred Std (%)': pred_std,
        'Actual Std (%)': np.std(actual_return) * 100,
    }


def normalise_weights(weights):
    weights = np.asarray(weights, dtype=float)

    if weights.sum() == 0:
        return np.ones(len(weights)) / len(weights)

    return weights / weights.sum()



def make_price_mae_blend(pred_matrix, data, future_col, start_weights=None):
    """Learn non-negative model weights that directly minimise validation price MAE.

    The model weights may sum to less than 1. The unused weight is the
    persistence baseline, whose predicted return is 0%.
    """
    pred_matrix = np.asarray(pred_matrix, dtype=float)

    current_price = data['Close'].to_numpy(dtype=float)
    actual_price = data[future_col].to_numpy(dtype=float)

    n_models = pred_matrix.shape[1]

    if start_weights is None:
        x0 = np.full(n_models, 0.8 / n_models)
    else:
        x0 = np.clip(
            np.asarray(start_weights, dtype=float),
            0.0,
            None,
        )

        if x0.sum() > 0:
            x0 = x0 / x0.sum() * 0.90
        else:
            x0 = np.full(n_models, 0.8 / n_models)

    def objective(weights):
        predicted_return = pred_matrix @ weights
        predicted_price = current_price * (1 + predicted_return)
        return np.mean(
            np.abs(actual_price - predicted_price)
        )

    constraints = [{
        'type': 'ineq',
        'fun': lambda weights: 1.0 - np.sum(weights),
    }]

    bounds = [
        (0.0, 1.0)
        for _ in range(n_models)
    ]

    result = minimize(
        objective,
        x0=x0,
        method='SLSQP',
        bounds=bounds,
        constraints=constraints,
        options={
            'maxiter': 300,
            'ftol': 1e-10,
            'disp': False,
        },
    )

    weights = np.clip(
        result.x,
        0.0,
        None,
    )

    if weights.sum() > 1.0:
        weights = weights / weights.sum()

    baseline_weight = max(
        0.0,
        1.0 - weights.sum(),
    )

    return weights, baseline_weight, result


def make_priority_nnls(pred_matrix, y, model_names):
    scores = []

    for i in range(len(model_names)):
        ic = np.corrcoef(pred_matrix[:, i], y)[0, 1]
        scores.append(max(ic, 0.001))

    scores = np.asarray(scores)
    scores[model_names.index('LightGBM')] *= LIGHTGBM_PRIORITY
    prior = scores / scores.sum()

    n_rows, n_models = pred_matrix.shape
    reg = np.sqrt(BLEND_PRIOR_STRENGTH)

    A = np.vstack([
        pred_matrix / np.sqrt(n_rows),
        reg * np.eye(n_models),
    ])

    b = np.concatenate([
        y / np.sqrt(n_rows),
        reg * prior,
    ])

    weights, _ = nnls(A, b)

    return normalise_weights(weights), prior


def build_models():
    return {
        'Ridge': Ridge(alpha=1.0),

        'Fast RBF-SVR': make_pipeline(
            Nystroem(
                kernel='rbf',
                gamma=0.05,
                n_components=100,
                random_state=42,
            ),
            LinearSVR(
                C=0.1,
                epsilon=0.005,
                dual=False,
                loss='squared_epsilon_insensitive',
                max_iter=2000,
                random_state=42,
            ),
        ),

        'LightGBM': lgb.LGBMRegressor(
            objective='regression_l1',
            n_estimators=1200,
            learning_rate=0.025,
            num_leaves=127,
            min_child_samples=40,
            subsample=0.90,
            colsample_bytree=0.90,
            reg_alpha=0.02,
            reg_lambda=0.05,
            random_state=42,
            n_jobs=-1,
            verbosity=-1,
        ),

        'MLP': MLPRegressor(
            hidden_layer_sizes=(64, 32),
            activation='relu',
            solver='adam',
            learning_rate_init=0.002,
            batch_size=1024,
            max_iter=35,
            early_stopping=True,
            validation_fraction=0.05,
            n_iter_no_change=4,
            random_state=42,
        ),

        'HistGradientBoosting': HistGradientBoostingRegressor(
            loss='squared_error',
            learning_rate=0.05,
            max_iter=300,
            max_leaf_nodes=63,
            min_samples_leaf=100,
            l2_regularization=0.1,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=20,
            random_state=42,
        ),

        'SGD-Huber': SGDRegressor(
            loss='huber',
            penalty='elasticnet',
            alpha=1e-5,
            l1_ratio=0.05,
            max_iter=300,
            tol=1e-4,
            average=True,
            random_state=42,
        ),
    }


def fit_model_set(X_raw, y):
    scaler = StandardScaler()

    X_scaled = scaler.fit_transform(
        X_raw
    ).astype(np.float32)

    models = build_models()

    models['Ridge'].fit(
        X_scaled,
        y,
    )

    models['Fast RBF-SVR'].fit(
        X_scaled,
        y,
    )

    models['LightGBM'].fit(
        X_raw,
        y,
    )

    models['MLP'].fit(
        X_scaled,
        y,
    )

    models['HistGradientBoosting'].fit(
        X_raw,
        y,
    )

    models['SGD-Huber'].fit(
        X_scaled,
        y,
    )

    return models, scaler


def prediction_matrix(models, scaler, X_raw, model_names):
    X_scaled = scaler.transform(
        X_raw
    ).astype(np.float32)

    predictions = {
        'Ridge':
            models['Ridge'].predict(X_scaled),

        'Fast RBF-SVR':
            models['Fast RBF-SVR'].predict(X_scaled),

        'LightGBM':
            models['LightGBM'].predict(X_raw),

        'MLP':
            models['MLP'].predict(X_scaled),

        'HistGradientBoosting':
            models['HistGradientBoosting'].predict(X_raw),

        'SGD-Huber':
            models['SGD-Huber'].predict(X_scaled),
    }

    return np.column_stack([
        predictions[name]
        for name in model_names
    ])


def confidence_percentile(predicted_return, reference_predictions):
    reference = np.sort(
        np.abs(
            np.asarray(reference_predictions)
        )
    )

    values = np.abs(
        np.asarray(predicted_return)
    )

    return (
        np.searchsorted(
            reference,
            values,
            side='right',
        )
        / len(reference)
        * 100
    )


def calibrate_display_signal(predicted_return, confidence_pct, max_boost=0.25):
    predicted_return = np.asarray(
        predicted_return,
        dtype=float,
    )

    confidence_pct = np.asarray(
        confidence_pct,
        dtype=float,
    )

    strength = np.clip(
        (confidence_pct - 50.0) / 50.0,
        0.0,
        1.0,
    )

    scale = (
        1.0
        + max_boost
        * (strength ** 1.5)
    )

    return predicted_return * scale


def predict_model_set(run, X_raw):
    return prediction_matrix(
        run['models'],
        run['scaler'],
        X_raw,
        run['model_names'],
    )


## 7. Train, blend and optimize validation price MAE


In [ ]:
all_runs = {}

for horizon in HORIZONS:
    print('\n' + '=' * 60)
    print(f'{horizon}-DAY HORIZON')
    print('=' * 60)

    target_col = f'Target_Return_{horizon}d'
    future_col = f'Future_Close_{horizon}d'

    ml_data = long_data.dropna(
        subset=feature_columns + [target_col, future_col]
    ).copy()

    ml_data = (
        ml_data
        .sort_values(['Date', 'Ticker'])
        .reset_index(drop=True)
    )

    train_data = ml_data[
        ml_data['Date'] < TRAIN_END
    ].copy()

    validation_data = ml_data[
        (ml_data['Date'] >= TRAIN_END) &
        (ml_data['Date'] < VALIDATION_END)
    ].copy()

    test_data = ml_data[
        ml_data['Date'] >= VALIDATION_END
    ].copy()

    X_train = train_data[feature_columns]
    y_train = train_data[target_col]

    X_val = validation_data[feature_columns]
    y_val = validation_data[target_col]

    X_test = test_data[feature_columns]
    y_test = test_data[target_col]

    # Train on the training period for model/blend selection.
    selection_models, selection_scaler = fit_model_set(
        X_train,
        y_train,
    )

    model_names = list(
        selection_models.keys()
    )

    val_matrix = prediction_matrix(
        selection_models,
        selection_scaler,
        X_val,
        model_names,
    )

    val_predictions = {
        name: val_matrix[:, i]
        for i, name in enumerate(model_names)
    }

    baseline_mae = mean_absolute_error(
        validation_data[future_col],
        validation_data['Close'],
    )

    results = [
        evaluate_predictions(
            'Baseline',
            np.zeros(len(validation_data)),
            validation_data,
            y_val,
            future_col,
            baseline_mae,
        )
    ]

    for name, pred in val_predictions.items():
        results.append(
            evaluate_predictions(
                name,
                pred,
                validation_data,
                y_val,
                future_col,
                baseline_mae,
            )
        )

    # Reference blend 1: standard NNLS on returns.
    nnls_weights, _ = nnls(
        val_matrix,
        y_val.to_numpy(),
    )

    nnls_weights = normalise_weights(
        nnls_weights
    )

    nnls_val_pred = (
        val_matrix @ nnls_weights
    )

    results.append(
        evaluate_predictions(
            'Blend (NNLS)',
            nnls_val_pred,
            validation_data,
            y_val,
            future_col,
            baseline_mae,
        )
    )

    # Reference blend 2: weak LightGBM-priority NNLS.
    priority_weights, prior = make_priority_nnls(
        val_matrix,
        y_val.to_numpy(),
        model_names,
    )

    priority_val_pred = (
        val_matrix @ priority_weights
    )

    results.append(
        evaluate_predictions(
            'Blend (LightGBM-priority)',
            priority_val_pred,
            validation_data,
            y_val,
            future_col,
            baseline_mae,
        )
    )

    # Final candidate:
    # optimise the exact validation metric we care about: future-price MAE.
    # The model weights are constrained >= 0 and may sum to < 1.
    # Any unused weight belongs to persistence (0% predicted return).
    mae_weights, baseline_weight, mae_opt_result = make_price_mae_blend(
        val_matrix,
        validation_data,
        future_col,
        start_weights=nnls_weights,
    )

    mae_val_pred = (
        val_matrix @ mae_weights
    )

    final_method = 'Price-MAE optimized + persistence'
    final_weights = mae_weights
    final_val_pred = mae_val_pred

    results.append(
        evaluate_predictions(
            f'Final Blend ({final_method})',
            final_val_pred,
            validation_data,
            y_val,
            future_col,
            baseline_mae,
        )
    )

    # After validation fixes the configuration and weights,
    # refit all base models on every row before the test period.
    pre_test_data = ml_data[
        ml_data['Date'] < VALIDATION_END
    ].copy()

    X_pre_test = pre_test_data[
        feature_columns
    ]

    y_pre_test = pre_test_data[
        target_col
    ]

    final_models, final_scaler = fit_model_set(
        X_pre_test,
        y_pre_test,
    )

    test_matrix = prediction_matrix(
        final_models,
        final_scaler,
        X_test,
        model_names,
    )

    final_test_pred = (
        test_matrix @ final_weights
    )

    test_baseline_mae = mean_absolute_error(
        test_data[future_col],
        test_data['Close'],
    )

    test_baseline_result = evaluate_predictions(
        'Baseline',
        np.zeros(len(test_data)),
        test_data,
        y_test,
        future_col,
        test_baseline_mae,
    )

    final_test_result = evaluate_predictions(
        f'Final Blend ({final_method}) — refit',
        final_test_pred,
        test_data,
        y_test,
        future_col,
        test_baseline_mae,
    )

    weights_df = pd.DataFrame({
        'Model': model_names + ['Persistence baseline'],
        'NNLS Weight': list(nnls_weights) + [0.0],
        'Priority Weight': list(priority_weights) + [0.0],
        'Final MAE Weight': list(final_weights) + [baseline_weight],
    })

    all_runs[horizon] = {
        'results': pd.DataFrame(results),
        'weights': weights_df,

        'models': final_models,
        'scaler': final_scaler,
        'model_names': model_names,

        'validation_data':
            validation_data.reset_index(drop=True),

        'y_validation':
            y_val.reset_index(drop=True),

        'final_validation_pred':
            final_val_pred,

        'test_data':
            test_data.reset_index(drop=True),

        'y_test':
            y_test.reset_index(drop=True),

        'final_test_pred':
            final_test_pred,

        'test_baseline_result':
            test_baseline_result,

        'final_test_result':
            final_test_result,

        'future_col':
            future_col,

        'target_col':
            target_col,

        'final_method':
            final_method,

        'final_weights':
            final_weights,

        'baseline_weight':
            baseline_weight,

        'mae_optimizer_success':
            mae_opt_result.success,

        'mae_optimizer_message':
            mae_opt_result.message,
    }

    print('\nValidation results')
    display(
        pd.DataFrame(results)
        .round(4)
    )

    print('\nBlend weights')
    display(
        weights_df.round(4)
    )

    print(
        f'\nPersistence weight: {baseline_weight:.4f}'
    )
    print(
        'MAE optimizer:',
        'success' if mae_opt_result.success else 'warning',
        '-',
        mae_opt_result.message,
    )
    print(
        f'Refit complete: final models trained on all data before {VALIDATION_END}.'
    )


## 8. Final validation results


In [ ]:
summary_rows = []

for horizon in HORIZONS:
    run = all_runs[horizon]
    final_name = f"Final Blend ({run['final_method']})"
    row = run['results'][run['results']['Model'] == final_name].iloc[0].to_dict()
    row['Horizon'] = f'{horizon} days'
    summary_rows.append(row)

final_validation_summary = pd.DataFrame(summary_rows)

columns = [
    'Horizon',
    'Model',
    'Price MAE ($)',
    'Price RMSE ($)',
    'Price MAPE (%)',
    'Direction Acc (%)',
    'Pearson IC',
    'Spearman IC',
    'MAE vs Baseline ($)',
    'Pred Std (%)',
    'Actual Std (%)',
]

display(final_validation_summary[columns].round(4))


## 9. Final ensemble weights

Unused ensemble weight is assigned to the persistence baseline (0% return).


In [ ]:
weight_rows = []

for horizon in HORIZONS:
    run = all_runs[horizon]

    for model, weight in zip(
        run['model_names'],
        run['final_weights'],
    ):
        weight_rows.append({
            'Horizon': f'{horizon} days',
            'Component': model,
            'Final Weight': weight,
        })

    weight_rows.append({
        'Horizon': f'{horizon} days',
        'Component': 'Persistence baseline',
        'Final Weight': run['baseline_weight'],
    })

final_weight_table = pd.DataFrame(weight_rows)

print('5D method:', all_runs[5]['final_method'])
print('10D method:', all_runs[10]['final_method'])
display(final_weight_table.round(4))


## 10. Confidence and display calibration


In [ ]:
confidence_outputs = {}
top_decile_rows = []
calibration_rows = []

for horizon in HORIZONS:
    run = all_runs[horizon]
    df = run['validation_data'][['Ticker', 'Date', 'Close']].copy()
    df['Predicted_Return'] = run['final_validation_pred']
    df['Actual_Return'] = run['y_validation'].to_numpy()

    df['Confidence_Percentile'] = confidence_percentile(
        df['Predicted_Return'].to_numpy(),
        run['final_validation_pred'],
    )
    df['Calibrated_Return'] = calibrate_display_signal(
        df['Predicted_Return'].to_numpy(),
        df['Confidence_Percentile'].to_numpy(),
    )

    df['Confidence_Decile'] = pd.qcut(
        df['Predicted_Return'].abs(),
        10,
        labels=False,
        duplicates='drop',
    )

    confidence = (
        df.groupby('Confidence_Decile')
        .apply(
            lambda g: pd.Series({
                'Rows': len(g),
                'Direction Acc (%)': (
                    np.sign(g['Predicted_Return']) ==
                    np.sign(g['Actual_Return'])
                ).mean() * 100,
                'Mean |Prediction| (%)': g['Predicted_Return'].abs().mean() * 100,
                'Mean |Calibrated| (%)': g['Calibrated_Return'].abs().mean() * 100,
            }),
            include_groups=False,
        )
        .reset_index()
    )

    confidence_outputs[horizon] = confidence

    top = confidence.iloc[-1]
    top_decile_rows.append({
        'Horizon': f'{horizon} days',
        'Top-decile Direction Acc (%)': top['Direction Acc (%)'],
        'Top-decile Raw |Prediction| (%)': top['Mean |Prediction| (%)'],
        'Top-decile Calibrated |Signal| (%)': top['Mean |Calibrated| (%)'],
    })

    raw_std = np.std(df['Predicted_Return']) * 100
    calibrated_std = np.std(df['Calibrated_Return']) * 100
    calibration_rows.append({
        'Horizon': f'{horizon} days',
        'Raw Pred Std (%)': raw_std,
        'Calibrated Display Std (%)': calibrated_std,
        'Display Std Increase (%)': (calibrated_std / raw_std - 1) * 100,
    })

    print(f'\n{horizon}-day confidence deciles')
    display(confidence.round(4))

print('\nTop-confidence summary')
display(pd.DataFrame(top_decile_rows).round(4))

print('\nDisplay calibration summary')
display(pd.DataFrame(calibration_rows).round(4))


## 11. Stock charts


In [ ]:
def plot_stock(ticker, horizon):
    run = all_runs[horizon]
    data = run['validation_data'].copy()

    data['Raw_Return'] = run['final_validation_pred']
    data['Actual_Return'] = run['y_validation'].to_numpy()
    data['Confidence_Percentile'] = confidence_percentile(
        data['Raw_Return'].to_numpy(),
        run['final_validation_pred'],
    )
    data['Calibrated_Return'] = calibrate_display_signal(
        data['Raw_Return'].to_numpy(),
        data['Confidence_Percentile'].to_numpy(),
    )

    data['Raw_Predicted_Price'] = data['Close'] * (1 + data['Raw_Return'])
    data['Calibrated_Price'] = data['Close'] * (1 + data['Calibrated_Return'])
    data['Correct'] = (
        np.sign(data['Raw_Return']) ==
        np.sign(data['Actual_Return'])
    )

    stock = data[data['Ticker'] == ticker].sort_values('Date').copy()
    if stock.empty:
        print('Ticker not found:', ticker)
        return

    future_col = run['future_col']

    plt.figure(figsize=(14, 5))
    plt.plot(
        stock['Date'],
        stock[future_col],
        color='black',
        linewidth=1.5,
        label='Actual future price',
    )
    plt.plot(
        stock['Date'],
        stock['Close'],
        color='gray',
        linestyle='--',
        linewidth=1,
        label='Baseline',
    )
    plt.plot(
        stock['Date'],
        stock['Raw_Predicted_Price'],
        linewidth=1,
        alpha=0.65,
        label='Raw final blend',
    )

    plt.scatter(
        stock.loc[stock['Correct'], 'Date'],
        stock.loc[stock['Correct'], 'Calibrated_Price'],
        color='green',
        s=22,
        label='Calibrated signal — correct direction',
    )
    plt.scatter(
        stock.loc[~stock['Correct'], 'Date'],
        stock.loc[~stock['Correct'], 'Calibrated_Price'],
        color='red',
        s=22,
        label='Calibrated signal — wrong direction',
    )

    plt.title(f'{ticker} — Final Blend — {horizon}-Day Prediction')
    plt.xlabel('Date')
    plt.ylabel('Price ($)')
    plt.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


for ticker in ['AAPL', 'MSFT', 'NVDA']:
    plot_stock(ticker, 5)
    plot_stock(ticker, 10)


## 12. Final out-of-sample test

The MAE-optimized weights are learned only from validation data, then frozen. Base models are refit on all pre-test data before this evaluation.


In [ ]:
test_rows = []

for horizon in HORIZONS:
    run = all_runs[horizon]

    baseline = dict(
        run['test_baseline_result']
    )

    baseline['Horizon'] = (
        f'{horizon} days'
    )

    baseline['Version'] = (
        'Baseline'
    )

    test_rows.append(
        baseline
    )

    raw = dict(
        run['final_test_result']
    )

    raw['Horizon'] = (
        f'{horizon} days'
    )

    raw['Version'] = (
        'Raw final blend'
    )

    test_rows.append(
        raw
    )

    test_data = run['test_data']
    y_test = run['y_test'].to_numpy()
    future_col = run['future_col']

    raw_pred = run['final_test_pred']

    confidence_pct = confidence_percentile(
        raw_pred,
        run['final_validation_pred'],
    )

    calibrated_pred = calibrate_display_signal(
        raw_pred,
        confidence_pct,
    )

    calibrated = evaluate_predictions(
        f'Calibrated display ({run["final_method"]})',
        calibrated_pred,
        test_data,
        y_test,
        future_col,
        run['test_baseline_result']['Price MAE ($)'],
    )

    calibrated['Horizon'] = (
        f'{horizon} days'
    )

    calibrated['Version'] = (
        'Calibrated display'
    )

    test_rows.append(
        calibrated
    )

final_test_table = pd.DataFrame(
    test_rows
)

columns = [
    'Horizon',
    'Version',
    'Model',
    'Price MAE ($)',
    'Price RMSE ($)',
    'Price MAPE (%)',
    'Direction Acc (%)',
    'Pearson IC',
    'Spearman IC',
    'MAE vs Baseline ($)',
    'Pred Std (%)',
    'Actual Std (%)',
]

display(
    final_test_table[
        columns
    ].round(4)
)


## 13. Latest stock signal


In [ ]:
def show_stock_signal(ticker):
    ticker = ticker.upper()
    rows = long_data[
        long_data['Ticker'].eq(ticker)
    ].dropna(subset=feature_columns).sort_values('Date')

    if rows.empty:
        print('Ticker not found:', ticker)
        return

    latest = rows.tail(1)
    current_price = float(latest['Close'].iloc[0])
    latest_date = latest['Date'].iloc[0]

    output = []

    for horizon in HORIZONS:
        run = all_runs[horizon]
        pred_matrix = predict_model_set(run, latest[feature_columns])
        raw_return = float(pred_matrix @ run['final_weights'])

        confidence_pct = float(
            confidence_percentile(
                np.array([raw_return]),
                run['final_validation_pred'],
            )[0]
        )
        calibrated_return = float(
            calibrate_display_signal(
                np.array([raw_return]),
                np.array([confidence_pct]),
            )[0]
        )

        if confidence_pct >= 90:
            confidence = 'High'
        elif confidence_pct >= 60:
            confidence = 'Medium'
        else:
            confidence = 'Low'

        output.append({
            'Horizon': f'{horizon} days',
            'Date': latest_date,
            'Current Price ($)': current_price,
            'Raw Model Return (%)': raw_return * 100,
            'Calibrated Signal (%)': calibrated_return * 100,
            'Raw Predicted Price ($)': current_price * (1 + raw_return),
            'Display Price ($)': current_price * (1 + calibrated_return),
            'Direction': 'Bullish' if raw_return > 0 else 'Bearish',
            'Confidence': confidence,
            'Confidence Percentile': confidence_pct,
        })

    display(pd.DataFrame(output).round(2))


show_stock_signal('AAPL')


## 14. Export evaluation and historical dashboard data

The UI uses these for historical results. New daily forecasts come from the saved models, not from the old snapshot CSV.


In [ ]:
# Export the files used by the Streamlit interface.

latest_rows = (
    long_data
    .dropna(subset=feature_columns)
    .sort_values(['Ticker', 'Date'])
    .groupby('Ticker', as_index=False)
    .tail(1)
    .reset_index(drop=True)
)

ui_data = latest_rows[['Ticker', 'Date', 'Close']].copy()
ui_data = ui_data.rename(columns={'Close': 'Current_Price'})

for horizon in HORIZONS:
    run = all_runs[horizon]

    pred_matrix = predict_model_set(
        run,
        latest_rows[feature_columns],
    )

    raw_pred = pred_matrix @ run['final_weights']

    confidence_pct = confidence_percentile(
        raw_pred,
        run['final_validation_pred'],
    )

    calibrated = calibrate_display_signal(
        raw_pred,
        confidence_pct,
    )

    ui_data[f'{horizon}D_Raw_Return'] = raw_pred
    ui_data[f'{horizon}D_Calibrated_Return'] = calibrated
    ui_data[f'{horizon}D_Confidence'] = confidence_pct

    ui_data[f'{horizon}D_Raw_Price'] = (
        ui_data['Current_Price'] * (1 + raw_pred)
    )

    ui_data[f'{horizon}D_Display_Price'] = (
        ui_data['Current_Price'] * (1 + calibrated)
    )


# Recent price history for the selected ticker.
price_history = (
    long_data[['Date', 'Ticker', 'Close']]
    .dropna()
    .sort_values(['Ticker', 'Date'])
    .groupby('Ticker', as_index=False, group_keys=False)
    .tail(300)
)


# Validation/test metrics, including calibrated values separately.
metric_rows = []

for horizon in HORIZONS:
    run = all_runs[horizon]

    validation_data = run['validation_data']
    y_val = run['y_validation'].to_numpy()
    future_col = run['future_col']

    validation_baseline = run['results'][
        run['results']['Model'].eq('Baseline')
    ].iloc[0].to_dict()

    validation_baseline['Horizon'] = horizon
    validation_baseline['Split'] = 'Validation'
    validation_baseline['Version'] = 'Baseline'
    metric_rows.append(validation_baseline)

    final_name = f"Final Blend ({run['final_method']})"
    validation_raw = run['results'][
        run['results']['Model'].eq(final_name)
    ].iloc[0].to_dict()

    validation_raw['Horizon'] = horizon
    validation_raw['Split'] = 'Validation'
    validation_raw['Version'] = 'Raw final blend'
    metric_rows.append(validation_raw)

    val_confidence = confidence_percentile(
        run['final_validation_pred'],
        run['final_validation_pred'],
    )

    val_calibrated = calibrate_display_signal(
        run['final_validation_pred'],
        val_confidence,
    )

    validation_calibrated = evaluate_predictions(
        f'Calibrated display ({run["final_method"]})',
        val_calibrated,
        validation_data,
        y_val,
        future_col,
        validation_baseline['Price MAE ($)'],
    )

    validation_calibrated['Horizon'] = horizon
    validation_calibrated['Split'] = 'Validation'
    validation_calibrated['Version'] = 'Calibrated display'
    metric_rows.append(validation_calibrated)

    test_baseline = dict(run['test_baseline_result'])
    test_baseline['Horizon'] = horizon
    test_baseline['Split'] = 'Out-of-sample test'
    test_baseline['Version'] = 'Baseline'
    metric_rows.append(test_baseline)

    test_raw = dict(run['final_test_result'])
    test_raw['Horizon'] = horizon
    test_raw['Split'] = 'Out-of-sample test'
    test_raw['Version'] = 'Raw final blend'
    metric_rows.append(test_raw)

    test_confidence = confidence_percentile(
        run['final_test_pred'],
        run['final_validation_pred'],
    )

    test_calibrated_pred = calibrate_display_signal(
        run['final_test_pred'],
        test_confidence,
    )

    test_calibrated = evaluate_predictions(
        f'Calibrated display ({run["final_method"]})',
        test_calibrated_pred,
        run['test_data'],
        run['y_test'].to_numpy(),
        future_col,
        test_baseline['Price MAE ($)'],
    )

    test_calibrated['Horizon'] = horizon
    test_calibrated['Split'] = 'Out-of-sample test'
    test_calibrated['Version'] = 'Calibrated display'
    metric_rows.append(test_calibrated)


# Confidence-decile behaviour.
confidence_export = pd.concat([
    confidence_outputs[h].assign(Horizon=h)
    for h in HORIZONS
], ignore_index=True)


# Final ensemble weights.
weights_export = final_weight_table.copy()


# Prediction history used by the UI.
# Keep only the most recent 80 rows per ticker/split/horizon so the file stays small.
prediction_frames = []

for horizon in HORIZONS:
    run = all_runs[horizon]
    future_col = run['future_col']

    for split_name, source_data, raw_pred, actual_return in [
        (
            'Validation',
            run['validation_data'],
            run['final_validation_pred'],
            run['y_validation'].to_numpy(),
        ),
        (
            'Out-of-sample test',
            run['test_data'],
            run['final_test_pred'],
            run['y_test'].to_numpy(),
        ),
    ]:
        frame = source_data[
            ['Ticker', 'Date', 'Close', future_col]
        ].copy()

        frame['Raw_Return'] = raw_pred
        frame['Actual_Return'] = actual_return

        frame['Confidence'] = confidence_percentile(
            raw_pred,
            run['final_validation_pred'],
        )

        frame['Calibrated_Return'] = calibrate_display_signal(
            raw_pred,
            frame['Confidence'].to_numpy(),
        )

        frame['Raw_Predicted_Price'] = (
            frame['Close'] * (1 + frame['Raw_Return'])
        )

        frame['Calibrated_Price'] = (
            frame['Close'] * (1 + frame['Calibrated_Return'])
        )

        frame['Actual_Future_Price'] = frame[future_col]

        frame['Correct_Direction'] = (
            np.sign(frame['Raw_Return']) ==
            np.sign(frame['Actual_Return'])
        )

        frame['Horizon'] = horizon
        frame['Split'] = split_name

        frame = (
            frame
            .sort_values(['Ticker', 'Date'])
            .groupby('Ticker', as_index=False, group_keys=False)
            .tail(80)
        )

        prediction_frames.append(
            frame[[
                'Ticker',
                'Date',
                'Close',
                'Horizon',
                'Split',
                'Raw_Return',
                'Calibrated_Return',
                'Confidence',
                'Raw_Predicted_Price',
                'Calibrated_Price',
                'Actual_Future_Price',
                'Actual_Return',
                'Correct_Direction',
            ]]
        )

prediction_history = pd.concat(
    prediction_frames,
    ignore_index=True,
)


ui_data.to_csv(
    'latest_signals.csv',
    index=False,
)

price_history.to_csv(
    'price_history.csv',
    index=False,
)

pd.DataFrame(metric_rows).to_csv(
    'model_metrics.csv',
    index=False,
)

confidence_export.to_csv(
    'confidence_deciles.csv',
    index=False,
)

weights_export.to_csv(
    'ensemble_weights.csv',
    index=False,
)

prediction_history.to_csv(
    'prediction_history.csv',
    index=False,
)

print('Saved: latest_signals.csv')
print('Saved: price_history.csv')
print('Saved: model_metrics.csv')
print('Saved: confidence_deciles.csv')
print('Saved: ensemble_weights.csv')
print('Saved: prediction_history.csv')


## Notes

- Validation chooses the ensemble weights.
- The final ensemble directly minimizes future-price MAE on validation data.
- All model weights are non-negative and their sum is allowed to be below 1.
- Any unused weight is automatically assigned to the persistence baseline (0% predicted return).
- The out-of-sample test is not used by the optimizer.
- After validation fixes the weights, all base models are refit on the full pre-test period.
- Official metrics use the raw final-blend prediction.
- The calibrated signal remains a separate display transformation for charts/UI.
- Confidence is based on prediction magnitude relative to validation predictions.
- No transaction costs, slippage, fundamentals, earnings or news are included.


## 15. Save models for automatic daily forecasts

Run this cell after training. Copy the generated `models` folder beside `app.py` if the notebook is run elsewhere.


In [ ]:
# Save the models needed for daily forecasting (run once after all training cells).
# The dashboard loads these files, so it does not need to retrain every day.
from pathlib import Path
import joblib

model_dir = Path('models')
model_dir.mkdir(exist_ok=True)

for horizon in HORIZONS:
    run = all_runs[horizon]
    kept = [name for name, weight in zip(run['model_names'], run['final_weights']) if weight > 1e-8]
    kept_weights = [float(weight) for name, weight in zip(run['model_names'], run['final_weights']) if weight > 1e-8]
    bundle = {
        'format_version': 1,
        'horizon': horizon,
        'trained_before': VALIDATION_END,
        'model_names': kept,
        'weights': kept_weights,
        'models': {name: run['models'][name] for name in kept},
        'scaler': run['scaler'],
        'features': list(feature_columns),
        'confidence_reference': np.sort(np.abs(np.asarray(run['final_validation_pred'], dtype=np.float32))),
        # The zero-return baseline is the unused share, so there is no model to save for it.
        'persistence_weight': float(run['baseline_weight']),
    }
    filename = model_dir / f'{horizon}d.joblib'
    joblib.dump(bundle, filename, compress=3)
    print(f'Saved {filename} with {len(kept)} active models ({filename.stat().st_size / 1e6:.1f} MB).')

# Save the same stock universe / sector mapping that the cross-sectional features use.
sector_lookup[['Ticker','Sector']].drop_duplicates().to_csv(model_dir / 'universe.csv', index=False)
print('Saved models/universe.csv')
